In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Carolina Miranda\Documents\Jose\bootcamp-data-science-portfolio\cases\case-spark-sql-queries\data\raw\ventas.csv")
print(df.columns.tolist())
df.head()

['id_venta', 'id_sucursal', 'producto', 'categoria', 'monto', 'fecha']


,id_venta,id_sucursal,producto,categoria,monto,fecha
0,1,784,Mochila,Accesorios,438201,2025-05-04
1,2,144,Mouse,Tecnologia,266498,2025-09-23
2,3,313,Monitor,Tecnologia,426047,2025-10-27
3,4,837,Teclado,Tecnologia,523122,2025-11-02
4,5,673,Impresora,Oficina,845609,2025-02-11


# Case: Consultas Distribuidas con Spark SQL

## Business Understanding
RetailMax necesita analizar sus transacciones de venta para identificar patrones por sucursal, categoría de producto y temporalidad. Estas consultas informan decisiones de inventario y asignación de recursos.

In [2]:
from pyspark.sql import SparkSession
from pathlib import Path

spark = SparkSession.builder \
    .appName("Case - Spark SQL Queries") \
    .master("local[*]") \
    .getOrCreate()

DATA_PATH = Path("../data/raw/ventas.csv")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [7]:
df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)
print(f"Total registros: {df.count()}")

root
 |-- id_venta: integer (nullable = true)
 |-- id_sucursal: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: integer (nullable = true)
 |-- fecha: date (nullable = true)

+--------+-----------+---------+----------+------+----------+
|id_venta|id_sucursal| producto| categoria| monto|     fecha|
+--------+-----------+---------+----------+------+----------+
|       1|        784|  Mochila|Accesorios|438201|2025-05-04|
|       2|        144|    Mouse|Tecnologia|266498|2025-09-23|
|       3|        313|  Monitor|Tecnologia|426047|2025-10-27|
|       4|        837|  Teclado|Tecnologia|523122|2025-11-02|
|       5|        673|Impresora|   Oficina|845609|2025-02-11|
+--------+-----------+---------+----------+------+----------+
only showing top 5 rows
Total registros: 1000000


In [8]:
df.createOrReplaceTempView("ventas")
print("Vista temporal 'ventas' registrada.")

Vista temporal 'ventas' registrada.


## SQL Queries
Tres consultas con filtros, agregaciones y ordenamientos.   

In [9]:
# Q1: Ventas de Tecnología con monto superior a 300,000
q1 = spark.sql("""
    SELECT id_venta, producto, monto, fecha
    FROM ventas
    WHERE categoria = 'Tecnologia'
      AND monto > 300000
    ORDER BY monto DESC
""")
q1.show()

+--------+--------+------+----------+
|id_venta|producto| monto|     fecha|
+--------+--------+------+----------+
|  477659|   Mouse|949999|2025-11-23|
|  211876| Monitor|949995|2025-12-27|
|  770426| Teclado|949992|2025-12-12|
|  747414| Monitor|949988|2025-04-21|
|  910346|   Mouse|949987|2025-10-16|
|   17418|   Mouse|949985|2025-10-31|
|  817455|   Mouse|949983|2025-05-25|
|   67848|   Mouse|949981|2025-09-10|
|  524294| Monitor|949980|2025-12-24|
|  292209|Notebook|949980|2025-07-17|
|  891683|   Mouse|949978|2025-01-15|
|  438597|Notebook|949977|2025-02-06|
|  737559|   Mouse|949977|2025-03-14|
|  964693|   Mouse|949976|2025-11-22|
|  899236| Teclado|949976|2025-03-12|
|  273300|Notebook|949975|2025-06-03|
|  407164| Monitor|949970|2025-11-13|
|  426123| Teclado|949969|2025-10-20|
|  693140|   Mouse|949968|2025-07-10|
|  275804| Monitor|949964|2025-09-12|
+--------+--------+------+----------+
only showing top 20 rows


In [10]:
# Q2: Ingreso total y cantidad de ventas por categoría
q2 = spark.sql("""
    SELECT categoria,
           COUNT(*) AS total_ventas,
           SUM(monto) AS ingreso_total,
           ROUND(AVG(monto), 0) AS ticket_promedio
    FROM ventas
    GROUP BY categoria
    ORDER BY ingreso_total DESC
""")
q2.show()

+----------+------------+-------------+---------------+
| categoria|total_ventas|ingreso_total|ticket_promedio|
+----------+------------+-------------+---------------+
|Tecnologia|      400538| 190436529608|       475452.0|
|   Oficina|      300156| 142972043268|       476326.0|
|    Utiles|      198955|  94521575965|       475090.0|
|Accesorios|      100351|  47749467444|       475825.0|
+----------+------------+-------------+---------------+



In [11]:
# Q3: Top 5 sucursales por ingreso total
q3 = spark.sql("""
    SELECT id_sucursal,
           COUNT(*) AS total_ventas,
           SUM(monto) AS ingreso_total
    FROM ventas
    GROUP BY id_sucursal
    ORDER BY ingreso_total DESC
    LIMIT 5
""")
q3.show()

+-----------+------------+-------------+
|id_sucursal|total_ventas|ingreso_total|
+-----------+------------+-------------+
|        375|        1105|    544630204|
|        494|        1100|    539565280|
|       1099|        1130|    526102951|
|        175|        1059|    520132402|
|        944|        1058|    519256304|
+-----------+------------+-------------+



In [12]:
# Plan de ejecución de Q1 para ver optimizaciones de Catalyst
print("=== Plan de ejecución Q1 ===")
q1.explain(True)

=== Plan de ejecución Q1 ===
== Parsed Logical Plan ==
'Sort ['monto DESC NULLS LAST], true
+- 'Project ['id_venta, 'producto, 'monto, 'fecha]
   +- 'Filter (('categoria = Tecnologia) AND ('monto > 300000))
      +- 'UnresolvedRelation [ventas], [], false

== Analyzed Logical Plan ==
id_venta: int, producto: string, monto: int, fecha: date
Sort [monto#21 DESC NULLS LAST], true
+- Project [id_venta#17, producto#19, monto#21, fecha#22]
   +- Filter ((categoria#20 = Tecnologia) AND (monto#21 > 300000))
      +- SubqueryAlias ventas
         +- View (`ventas`, [id_venta#17, id_sucursal#18, producto#19, categoria#20, monto#21, fecha#22])
            +- Relation [id_venta#17,id_sucursal#18,producto#19,categoria#20,monto#21,fecha#22] csv

== Optimized Logical Plan ==
Sort [monto#21 DESC NULLS LAST], true
+- Project [id_venta#17, producto#19, monto#21, fecha#22]
   +- Filter ((isnotnull(categoria#20) AND isnotnull(monto#21)) AND ((categoria#20 = Tecnologia) AND (monto#21 > 300000)))
      +- R

## Cómo Catalyst optimiza estas consultas

1. **Parsed Logical Plan**: Parsea el SQL y construye un árbol lógico sin resolver.
2. **Analyzed Logical Plan**: Resuelve nombres de columnas y tipos contra el catálogo.
3. **Optimized Logical Plan**: Aplica optimizaciones como *predicate pushdown* (empuja `WHERE` lo más cerca posible de la fuente) y *projection pruning* (elimina columnas no necesarias).
4. **Physical Plan**: Selecciona la estrategia de ejecución más eficiente (sort, hash, broadcast).

En nuestro Q1, Catalyst empuja el filtro `categoria = 'Tecnologia' AND monto > 300000` antes del ordenamiento, reduciendo los datos que se procesan en etapas posteriores.

## Insight → Possible Decision
- **Q1**: Identifica productos de alto valor en Tecnología → decisión de stock prioritario.
- **Q2**: Revela qué categorías generan mayor ingreso → asignación de presupuesto de marketing.
- **Q3**: Muestra las sucursales top → potencial benchmark de mejores prácticas hacia sucursales de menor rendimiento.

In [13]:
spark.stop()
print("Sesión Spark cerrada.")

Sesión Spark cerrada.
